<img src="http://imgur.com/1ZcRyrc.png" style="float: left; margin: 20px; height: 55px">

# Introduction to Regularization


> Authors: Kiefer Katovich + Justin Pounders + Matt Brems

In [ ]:
!pip install ipywidgets IPython

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('darkgrid')
%config InlineBackend.figure_format = 'retina'
%matplotlib inline

## Agenda
- Review
- Overview of Regularization
- Load Data/Basic EDA
- Standardizing Predictors
- Visualizing Regularization
- Implementing Regularization

## Review

<details><summary>What is the bias-variance tradeoff?</summary>

- Mean squared error can be decomposed into a bias component plus a variance component (plus a systematic error, but we don't have control over this part, so we often ignore it).
- The bias-variance tradeoff refers to the fact that taking steps to minimize bias usually comes at the expense of an increase in variance. Similarly, taking steps to minimize variance usually comes at the expense of an increase in bias.

</details>

---

<details><summary>What evidence/information would lead me to believe that my model suffers from high variance?</summary>
    
- After splitting my data into training and testing sets, if I see that my model performs way better on my training set than my testing set, this means that my model is not generalizing very well to "new" data.
- An example might be where our training MSE is substantially lower than our testing MSE, or where our training R-squared is substantially higher than our testing R-squared.
</details>

---

<details><summary>What term do we use to describe a model that suffers from high variance?</summary>

- Overfit.
</details>

<a id='intro'></a>

## Why is high variance bad?

High variance is bad because it means that our model doesn't generalize well to new data. This means that our model looks as though it performs well on our training data but won't perform as well on new, unseen data.

---
<details><summary>How might we try to fix a model that suffers from high variance?</summary>

- Drop features.
- Make our existing features less complex. (i.e. get rid of interaction terms or higher order terms.)
- Choose a simpler model.
- Gather more data. (Although this is usually expensive and time-consuming.)
- Regularization!
</details>

## Overview of regularization

---

The goal of "regularizing" regression models is to **structurally (automatically) prevent overfitting** by imposing a penalty while fitting the model.

Remember what our loss function (so far!) looks like:

$$
\begin{eqnarray*}
\text{minimize: }SSE &=& \sum_{i=1}^n (y_i - \hat{y}_i)^2 \\
&=& (\mathbf{y} - \mathbf{X}\pmb\beta)^T (\mathbf{y} - \mathbf{X}\pmb\beta)
\end{eqnarray*}
$$

- Machine learning usually boils down to _optimization_.
- Training/fitting means _minimizing_ a "loss function."
- In ordinary least squares (OLS) linear regression, the loss function is SSE, which gives you the optimal $\beta$ values for your regression model!

Regularization adds an extra piece to the OLS loss function. We now minimize this new loss function, which is the sum of squared errors plus a penalty.

$$
\begin{eqnarray*}
\text{minimize: }SSE + \text{Ridge penalty} &=& \sum_{i=1}^n (y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^p \beta_j^2
\end{eqnarray*}
$$

#### What does this do?
This new loss function ensures the coefficients don't get too big when they shouldn't!

Put another way, this new loss function makes sure our coefficients are based on signal (the thing we want to detect/understand) and not noise (the random fluctuations in our data we want to ignore).
- If the coefficients are large because of noise, then regularization "detects this" and pushes the coefficients back toward zero.
- If the coefficients are large because they're supposed to be, regularization will have little effect on the coefficients.

The two most common types of regularization are the **Ridge** and **Lasso**. There is also mixture of them called the **Elastic Net**.

## What is the effect of regularization?

---

**To demonstrate the effects we will be using a dataset on wine quality.**

### Load the wine csv

This version has red and white wines concatenated together and tagged with a binary 1,0 indicator (1 is red wine). There are many other variables purportedly related to the rated quality of the wine.

In [ ]:
# Load in the wine .csv.
wine = 

# Convert all columns to lowercase and replace spaces in column names.
wine.columns = 

In [ ]:
# Check the first five rows.


In [ ]:
# Check for missing values.


An important aspect of this data is that there is **multicollinearity** in the data. The term multicollinearity means that there are _high correlations between predictor variables in your model_. 
- Remember: one of the assumptions of a multiple linear regression model is that our independent variables are independent of one another!

**Multicollinearity can lead to a variety of problems**:
1. Small changes in your training data can result in very different model fits.
2. With very high multicollinearity, the inverse matrix the computer calculates may not be accurate.
    - $\hat{\pmb\beta} = (\mathbf{X}^T\mathbf{X})^{-1}(\mathbf{X}^T\mathbf{y})$ may throw an error like "Your condition number is large" or "your coefficient estimates are not stable."
3. Interpreting the coefficient of a variable that is strongly correlated with another variable is tricky.
    - **For example:** Say I want to predict income using age and years of professional experience as predictors. Attempting to interpret a one-unit increase in age might not make sense, because age and years of professional experience are probably pretty highly correlated with one another.

In [ ]:
# Look at the correlation between variables.

wine_corr = 
wine_corr

In [ ]:
# Generate a heatmap in seaborn.

# Set the default matplotlib figure size to 12x12:
fix, ax = plt.subplots(figsize=(12,12))

# Generate a mask for the upper triangle (taken from seaborn example gallery)
mask = np.zeros_like(wine_corr, dtype=np.bool)
mask[np.triu_indices_from(mask)] = True

# Plot the heatmap with seaborn.
# Assign the matplotlib axis the function returns. This will let us resize the labels.
ax = sns.heatmap(wine_corr, mask=mask, ax=ax)

# Resize the labels.
ax.set_xticklabels(ax.xaxis.get_ticklabels(), fontsize=14)
ax.set_yticklabels(ax.yaxis.get_ticklabels(), fontsize=14)

# If you put plt.show() at the bottom, it prevents those useless printouts from matplotlib.
plt.show()

#### Note: At this point, we'd usually split our data up into training and testing sets. We won't explicitly do this for this activity, but in general you would want to split your data into training and testing sets before fitting models!

### Create the target and predictor matrix 

We will be making two X, Y datasets. One of them will be dramatically more complex (and thus overfit).

1. The first is going to just be all the variables added together.
2. The second will have all interactions between the variables included. This should overfit the target quite a bit!

#### With sklearn

In [ ]:
# select all the columns that are not the target
nc = [x for x in wine.columns if x != 'quality']

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

X =
y =

pf = 
pf = 
Xoverfit = 

**Look at how the shapes of the X matrices differ. You can see that the Xoverfit has considerably more columns due to the creation of all possible two-way interactions.**

In [ ]:
print(X.shape)
print(Xoverfit.shape)

#### Let's split our data up into training and testing sets. Why do we split our data into training and testing sets?

In [ ]:
# import train/test/split


In [ ]:
print('X_train shape is: ' + str(X_train.shape))
print('y_train shape is: ' + str(y_train.shape))
print('X_test shape is: ' + str(X_test.shape))
print('y_test shape is: ' + str(y_test.shape))

In [ ]:
print('X_overfit_train shape is: ' + str(X_overfit_train.shape))
print('y_overfit_train shape is: ' + str(y_overfit_train.shape))
print('X_overfit_test shape is: ' + str(X_overfit_test.shape))
print('y_overfit_test shape is: ' + str(y_overfit_test.shape))

In [ ]:
y_train.head()

In [ ]:
y_overfit_train.head()

<a id='standardization'></a>

## Standardizing predictors is required

---

When regularizing, **it is neccessary to standardize the predictor columns before constructing the models, even the dummy coded categorical variables**.

Below we define our target variable, then normalize the columns that are not the target.

We are going to change our loss function to add a penalty. The equations for the Ridge and Lasso penalties:

### $$ \text{Ridge penalty}\; = \alpha \sum_{j=1}^p \beta_j^2$$

### $$ \text{Lasso penalty}\; = \alpha \sum_{j=1}^p |\beta_j|$$

<details><summary>Why might I need to standardize my variables before regularizing?</summary>
    
- Regularization penalizes the magnitude (size) of the coefficients.
- The size of the coefficient depends on the scale of the independent (X) variable.
- Suppose I wanted to predict one's income using their high school GPA and the square footage of their childhood home as predictors. 
$$
\begin{eqnarray*}
\text{income} &=& \beta_0 + \beta_1\text{GPA} + \beta_2\text{sqft}
\end{eqnarray*}
$$
</details>

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
# Instantiate Standard Scaler.


# Fit Standard Scaler to data.


# Transform data using Standard Scaler.


In [ ]:
Z_train.head()

In [ ]:
# Instantiate Standard Scaler.


# Fit Standard Scaler to overfit data AND transform in one step.


In [ ]:
Z_overfit_train.head()

<a id='viz-ridge'></a>

## Defining and Visualizing the Ridge

---

### $$ \text{minimize:}\; SSE+Ridge = \sum_{i=1}^n \left(y_i - \hat{y}_i\right)^2 + \alpha\sum_{j=1}^p \beta_j^2$$

**Where:**

$\beta_j^2$ is the squared coefficient for variable $x_j$.

$\sum_{j=1}^n \beta_j^2$ is the sum of these squared coefficients for every variable in the model. 
- This does **not** include the intercept $\beta_0$.

$\alpha$ is a constant for the _strength_ of the regularization parameter. The higher the value, the greater the impact of this new component in the loss function. If the value was zero, we would revert back to just the least squares loss function. If the value was a billion, however, the residual sum of squares component would have a much smaller effect on the loss/cost than the regularization term.

Now all **you** have to do is...

import the `Ridge` model class from sklearn.

In [ ]:
from sklearn.linear_model import Ridge

This function iterates over a series of different alpha regularization parameters and stores the results of the model so that we can plot them interactively.

> This will not be part of your typical workflow.  This for learning purposes only!

**Alpha values for the ridge are best visualized on a logarithmic "magnitude" scale.** Essentially, the effect of alpha on the coefficients does not increase linearly but by orders of magnitude. 

In [ ]:
def ridge_coefs(X, y, alphas):
    
    # set up the list to hold the different sets of coefficients:
    coefs = []
    
    # Set up a ridge regression object
    ridge_reg = Ridge()
    
    # Iterate through the alphas fed into the function:
    for a in alphas:
        
        # On each alpha reset the ridge model's alpha to the current one:
        ridge_reg.set_params(alpha=a)
        
        # fit or refit the model on the provided X, y
        ridge_reg.fit(X, y)
        
        # Get out the coefficient list
        coefs.append(ridge_reg.coef_)
        
    return coefs

# np.logspace gives us points between specified orders of magnitude on a logarithmic scale. It is base 10.
r_alphas = np.logspace(0, 5, 200)

# Get the coefficients for each alpha for the Ridge, using the function above
r_coefs = ridge_coefs(X, y, r_alphas)

# The cycler package lets us "cycle" through colors.
# Just another thing I had to look up on stackoverflow. That's my life.

from cycler import cycler

def coef_plotter(alphas, coefs, feature_names, to_alpha, regtype='ridge'):
    
    # Get the full range of alphas before subsetting to keep the plots from 
    # resetting axes each time. (We use these values to set static axes later).
    amin = np.min(alphas)
    amax = np.max(alphas)
    
    # Subset the alphas and coefficients to just the ones below the set limit
    # from the interactive widget:
    alphas = [a for a in alphas if a <= to_alpha]
    coefs = coefs[0:len(alphas)]
    
    # Get some colors from seaborn:
    colors = sns.color_palette("husl", len(coefs[0]))
    
    # Get the figure and reset the size to be wider:
    fig = plt.figure()
    fig.set_size_inches(18,5)

    # We have two axes this time on our figure. 
    # The fig.add_subplot adds axes to our figure. The number inside stands for:
    #[figure_rows|figure_cols|position_of_current_axes]
    ax1 = fig.add_subplot(121)
    
    # Give it the color cycler:
    ax1.set_prop_cycle(cycler('color', colors))
    
    # Print a vertical line showing our current alpha threshold:
    ax1.axvline(to_alpha, lw=2, ls='dashed', c='k', alpha=0.4)
    
    # Plot the lines of the alphas on x-axis and coefficients on y-axis
    ax1.plot(alphas, coefs, lw=2)
    
    # set labels for axes:
    ax1.set_xlabel('alpha', fontsize=20)
    ax1.set_ylabel('coefficients', fontsize=20)
    
    # If this is for the ridge, set this to a log scale on the x-axis:
    if regtype == 'ridge':
        ax1.set_xscale('log')
    
    # Enforce the axis limits:
    ax1.set_xlim([amin, amax])
    
    # Put a title on the axis
    ax1.set_title(regtype+' coef paths\n', fontsize=20)
    
    # Get the ymin and ymax for this axis to enforce it to be the same on the 
    # second chart:
    ymin, ymax = ax1.get_ylim()

    # Add our second axes for the barplot in position 2:
    ax2 = fig.add_subplot(122)
    
    # Position the bars according to their index from the feature names variable:
    ax2.bar(list(range(1, len(feature_names)+1)), coefs[-1], align='center', color=colors)
    ax2.set_xticks(list(range(1, len(feature_names)+1)))
    
    # Reset the ticks from numbers to acutally be the names:
    ax2.set_xticklabels(feature_names, rotation=65, fontsize=12)
    
    # enforce limits and add titles, labels
    ax2.set_ylim([ymin, ymax])
    ax2.set_title(regtype+' predictor coefs\n', fontsize=20)
    ax2.set_xlabel('coefficients', fontsize=20)
    ax2.set_ylabel('alpha', fontsize=20)
    
    plt.show()

The plotting function above will:

 - Plot the effect of changing alpha on the coefficient size on a **path** graph
 - Plot the effect of changing alpha on the coefficient size on a **bar** graph
 
Each one gives informative information. It's just two different ways of visualizing the same thing. The chart is interactive so you can play around with the values of alpha across the specified range above.

The function and `interact` from ipywidgets lets me take some specified alphas that we have already calculated the coefficients for and plot them out in an interactive way.

In [ ]:
from ipywidgets import *
from IPython.display import display

def ridge_plot_runner(log_of_alpha=0):
    coef_plotter(r_alphas, r_coefs, X.columns, 10**log_of_alpha, regtype='ridge')

interact(ridge_plot_runner, log_of_alpha=(0.1,5,0.2))

<a id='viz-lasso'></a>

## Defining and Visualizing the Lasso

---

Now we do the same thing as above but for the Lasso.

Lasso regression takes a different approach. Instead of adding the sum of squared $\beta$ coefficients to the SSE, it adds the sum of the absolute values of the $\beta$ coefficients:

### $$ \text{minimize:}\; SSE + Lasso = \sum_{i=1}^n \left(y_i - \hat{y}_i\right)^2 + \alpha\sum_{j=1}^p |\beta_j|$$

**Where:**

$|\beta_j|$ is the absolute value of the $\beta$ coefficient for variable $x_j$.

$\alpha$ is again the strength of the regularization penalty component in the loss function. 

In [ ]:
from sklearn.linear_model import Lasso

In [ ]:
# This is the same as the ridge coefficient by alpha calculator
def lasso_coefs(X, y, alphas):
    coefs = []
    lasso_reg = Lasso()
    for a in alphas:
        lasso_reg.set_params(alpha=a)
        lasso_reg.fit(X, y)
        coefs.append(lasso_reg.coef_)
    return coefs


Alphas for the Lasso tend to effect regularization linearly rather than by orders of magnitude like in the ridge. 

A linear series of alphas is sufficient.

In [ ]:
l_alphas = np.arange(0.001, 0.15, 0.0025)
l_coefs = lasso_coefs(X, y, l_alphas)

Run the same plotting function above, but now with the calculated coefficients of alpha for the Lasso.

In [ ]:
def lasso_plot_runner(alpha=0):
    coef_plotter(l_alphas, l_coefs, X.columns, alpha, regtype='lasso')

interact(lasso_plot_runner, alpha=(0.001,0.2,0.01))

<a id='viz-elastic-net'></a>

## Visualizing the Elastic Net

---

Can't decide?  Buy both!!

The Elastic Net combines the Ridge and Lasso penalties.  It adds *both* penalties to the loss function:

### $$ \text{minimize:}\; SSE + Ridge + Lasso = \sum_{i=1}^n \left(y_i - \hat{y}_i\right)^2 + \alpha\left[\rho\sum_{j=1}^p |\beta_j| + (1-\rho)\sum_{j=1}^p \beta_j^2\right]$$

In the elastic net, the effect of the ridge versus the lasso is balanced by the $\rho$ parameter.  It is the ratio of Lasso penalty to Ridge penalty and must be between zero and one.

`ElasticNet` in sklearn has two parameters:
- `alpha`: the regularization strength.
- `l1_ratio`: the amount of L1 vs L2 penalty (i.e., $\rho$). An l1_ratio of 0 is equivalent to the Ridge, whereas an l1_ratio of 1 is equivalent to the Lasso.


In [ ]:
from sklearn.linear_model import ElasticNet

Calculate the coefficients with both alpha values and an l1_ratio of 0.05. Lasso can "overpower" the Ridge penalty in some datasets, and so rather than an equal balance I'm just adding a little bit of Lasso in.
- Using a $\rho$ value below 0.05 can empirically cause issues in `sklearn`.

In [ ]:
# This is the same as the ridge coefficient by alpha calculator
def elasticnet_coefs(X, y, alphas):
    coefs = []
    enet_reg = ElasticNet()
    for a in alphas:
        enet_reg.set_params(alpha=a, l1_ratio=0.05)
        enet_reg.fit(X, y)
        coefs.append(enet_reg.coef_)
        
    return coefs

In [ ]:
enet_alphas = np.arange(0.01, 1.0, 0.005)
enet_coefs = elasticnet_coefs(X, y, enet_alphas)

In [ ]:
def enet_plot_runner(alpha=0.01):
    coef_plotter(enet_alphas, enet_coefs, X.columns, alpha, regtype='elastic net')

interact(enet_plot_runner, alpha=(0.0,1.0,0.005))

<a id='model-performance'></a>

## Model performance of the Ridge, Lasso and Elastic Net on the overfit data

### (i.e., what do I actually need to do?)

---

Let's check out how the penalties affect the performance of our models. On the basic wine dataset there won't be much benefit to using regularization. We can switch over to the overfit data instead to see if regularization helps us control overfitting.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score

**Let's start with plain vanilla linear regression:**

In [ ]:
# Instantiate linear regression model without regularization.
lr_model =

In [ ]:
# Fit linear regression model.


In [ ]:
# Generate predictions from linear regression model.


In [ ]:
# Evaluate model.
# Let's estimate what our model's R2 might be on new, out-of-sample data.


<details><summary>How do we interpret this $R^2$ score?</summary>
32% of the variability in our $Y$ (wine quality) is explained by our model, relative to a baseline model with no predictors.
</details>

In [ ]:
# Let's use R2 to see if our model is overfit, underfit, or neither.


<details><summary>Given these two R2 scores together, what do you conclude?</summary>
Because our training performance is better than our testing performance, it's likely that our model is overfit (and thus too complex). We should consider regularizing.
</details>

### Ridge Regression

**Now let's make it into a Ridge regression by adding the penalty term:**

In [ ]:
# Instantiate.
ridge_model =

# Fit.


# Generate predictions.


# Evaluate model using R2.


Which $\alpha$ to choose?  Let's search! (Remember that we search logarithmic space for Ridge regularization.)

In [ ]:
from sklearn.linear_model import RidgeCV

In [ ]:
# Set up a list of ridge alphas to check.
r_alphas = 
# Generates 200 values equally between 0 and 5,
# then converts them to alphas between 10^0 and 10^5.

# Cross-validate over our list of ridge alphas.
ridge_model = 

# Fit model using best ridge alpha!
ridge_model = 

In [ ]:
# Here is the optimal value of alpha
ridge_optimal_alpha = 
ridge_optimal_alpha

Let's get the corresponding value of $R^2$.

In [ ]:
# Instantiate model.
ridge_opt = 

# Fit model.
ridge_opt

# Generate predictions
ridge_opt_preds =
ridge_opt_preds_train =

# Evaluate model.


### Lasso Regression

In [ ]:
from sklearn.linear_model import Lasso, LassoCV

In [ ]:
# Set up a list of Lasso alphas to check.
l_alphas = 
# Generates values equally between 0.15 and 1 in steps of 0.01.

# Cross-validate over our list of Lasso alphas.
lasso_model = 

# Fit model using best ridge alpha!
lasso_model = 

In [ ]:
# Here is the optimal value of alpha
lasso_optimal_alpha = 
lasso_optimal_alpha

In [ ]:
# Generate predictions


# Evaluate model.


### Let's check out the coefficients of the Lasso and Ridge models.

### ElasticNet Regression

In [ ]:
from sklearn.linear_model import ElasticNet, ElasticNetCV

In [ ]:
# Set up a list of alphas to check.
enet_alphas = np.arange(0.5, 1.0, 0.005)

# Set up our l1 ratio. (What does this do?)
enet_ratio = 0.5

# Instantiate model.
enet_model = ElasticNetCV(alphas=enet_alphas, l1_ratio=enet_ratio, cv=5)

# Fit model using optimal alpha.
enet_model = enet_model.fit(X_overfit_train, y_overfit_train)

# Generate predictions.
enet_model_preds = enet_model.predict(X_overfit_test)
enet_model_preds_train = enet_model.predict(X_overfit_train)

# Evaluate model.
print(r2_score(y_overfit_test, enet_model_preds))
print(r2_score(y_overfit_train, enet_model_preds_train))

In [ ]:
# Here is the optimal value of alpha.
enet_optimal_alpha = enet_model.alpha_
enet_optimal_alpha

### Recap

<details><summary>In regularization, when do we standardize our variables?Which variables do we standardize?</summary>
    
- We always standardize all independent variables!
</details>

<details><summary>In regularization, what role does $\alpha$ play?</summary>

- $\alpha$ determines how strongly we want to regularize. 
- As $\alpha$ increases, we regularize more, so our coefficients will shrink toward zero.
- As $\alpha$ decreases, we regularize less, so our coefficients will grow away from zero.
</details>

<details><summary>Which type of regularization might I use as a feature selection method? Why?</summary>
    
- Lasso. Because we regularize the absolute value of the coefficients, some coefficients will likely be "zeroed" out. (That is, their $\beta$ values go to 0.)
</details>

<details><summary>When searching for an optimal value of $\alpha$, should I search over a linear space or a logarithmic space? </summary>
    
- For Ridge, we should search over a logarithmic space. The effect of $\alpha$ is based on orders of magnitude.
- For Lasso, we should search over a linear space. The effect of $\alpha$ is linear.
</details>